# ツールの比較（Google Colab）

meikey と、同じように ed25519 の SSH 公開鍵に好きな文字列を入れるツールをビルドして、このマシンで同じ秒数ずつ速さを測ります。上のセルから順に実行してください。

- 測るツールと測り方は [ツールの比較](https://github.com/waxsd100/meikey/blob/master/docs/comparison.ja.md) にあります
- GPU は **ランタイム → ランタイムのタイプを変更** で選びます（既定は G4）。GPU が無いと、GPU を使うツールは飛ばします
- ビルドに 20〜40 分ほど、測るのに「ツールの数 × 秒数」ほどかかります
- 使い終わったら **ランタイム → ランタイムを接続解除して削除** してください

## 1. マシンを確認する

In [ ]:
#@title GPU・CPU・メモリを表示する
import os, subprocess

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    return r.stdout.strip()

print(sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "GPU なし")
print(sh("LC_ALL=C lscpu | grep -E '^(Model name|CPU\\(s\\)|Thread|Core)'"))
print(sh("free -g | head -2"))

## 2. ビルドに使うものを入れる

C・C++ のライブラリ（libsodium・OpenCL）、Rust、Go、Python のパッケージを入れます。NVIDIA の GPU があれば、OpenCL から使えるようにします。

In [ ]:
import glob, json, os, urllib.request

WORK = "/content/compare"
os.makedirs(WORK, exist_ok=True)

!apt-get -qq update > /dev/null && apt-get -qq install -y libsodium-dev ocl-icd-opencl-dev opencl-headers clinfo > /dev/null

# NVIDIA のドライバの OpenCL を、OpenCL のローダーに登録する
libs = sorted(glob.glob("/usr/lib64-nvidia/libnvidia-opencl.so*") + glob.glob("/usr/lib/x86_64-linux-gnu/libnvidia-opencl.so*"))
if libs:
    os.makedirs("/etc/OpenCL/vendors", exist_ok=True)
    with open("/etc/OpenCL/vendors/nvidia.icd", "w") as f:
        f.write(libs[0] + "\n")
!clinfo -l

# Rust（ほかのツールは stable、meikey は rust-toolchain.toml の版でビルドする）
CARGO_BIN = os.path.expanduser("~/.cargo/bin")
if not os.path.exists(os.path.join(CARGO_BIN, "cargo")):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal --default-toolchain stable
os.environ["PATH"] = CARGO_BIN + ":" + os.environ["PATH"]

# Go（最新の版）
if not os.path.exists(f"{WORK}/go/bin/go"):
    ver = json.load(urllib.request.urlopen("https://go.dev/dl/?mode=json"))[0]["version"]
    !curl -sSL https://go.dev/dl/{ver}.linux-amd64.tar.gz | tar -C {WORK} -xz
!{WORK}/go/bin/go version
!rustc --version

## 3. meikey と、比べるためのスクリプトを取ってくる

In [ ]:
import sys

REPO = "https://github.com/waxsd100/meikey"
BRANCH = "master"
if os.path.exists("/content/meikey"):
    !git -C /content/meikey pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO} /content/meikey
!cd /content/meikey && rustup toolchain install

sys.path.insert(0, "/content/meikey/colab")
import importlib, compare
importlib.reload(compare)
print("\n".join(f"{b.name}: {b.url}" for b in compare.BUILDS))

## 4. ツールをビルドする

GPU が無いときと、CUDA Toolkit（nvcc）が無いときは、GPU 専用のツールを飛ばします。ログは `/content/compare/logs/` にあります。

In [ ]:
built = compare.build_all()

## 5. 速さを測る

ツールを1つずつ、同じ秒数だけ動かして測ります。測っている間は、ほかのツールは動かしません。

In [ ]:
#@title 測る
SECONDS = 30  #@param {type:"integer"}

import pandas as pd

results = compare.bench_all(SECONDS, built)
table = pd.DataFrame(results).sort_values("鍵/秒", ascending=False, na_position="last")
display(table.drop(columns=["鍵/秒"]).reset_index(drop=True))

## 6. 結果を保存する

In [ ]:
from google.colab import files

gpu = sh("nvidia-smi --query-gpu=name --format=csv,noheader") or "GPU なし"
cpu = sh("LC_ALL=C lscpu | sed -n 's/^Model name: *//p'")
with open(f"{WORK}/results.md", "w") as f:
    f.write(f"- GPU: {gpu}\n- CPU: {cpu}（{os.cpu_count()} スレッド）\n- 1つあたり {SECONDS} 秒\n\n")
    f.write(compare.markdown(results) + "\n")
table.to_csv(f"{WORK}/results.csv", index=False)
print(open(f"{WORK}/results.md").read())
files.download(f"{WORK}/results.md")
files.download(f"{WORK}/results.csv")